In [3]:
pip install langchain chromadb groq tiktoken pypdf langchain-community langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

In [6]:
from langchain.schema import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [7]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [9]:
vector_store = Chroma(
    embedding_function=HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    ),
    persist_directory="my_chroma_db",
    collection_name="sample"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\Anushka\AppData\Local\Temp\ipykernel_33228\2089461126.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(


In [10]:
# add documents
vector_store.add_documents(docs)

['ef337d33-f0e0-4d55-85c9-45fb51bc85fb',
 '52eca65f-aa5a-492b-a796-cc707f241e31',
 '018e8cd7-4f38-48b3-98f5-8b08a63df2c9',
 '94c548ec-ff3f-428c-bea3-638a937c1f21',
 'd033685e-12e3-468a-bf1b-4d44d627b1c3']

In [11]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['ef337d33-f0e0-4d55-85c9-45fb51bc85fb',
  '52eca65f-aa5a-492b-a796-cc707f241e31',
  '018e8cd7-4f38-48b3-98f5-8b08a63df2c9',
  '94c548ec-ff3f-428c-bea3-638a937c1f21',
  'd033685e-12e3-468a-bf1b-4d44d627b1c3'],
 'embeddings': array([[ 0.00994723,  0.06914338, -0.05147114, ..., -0.0354334 ,
          0.01284808,  0.01248293],
        [ 0.00127743,  0.03129851, -0.02375379, ..., -0.00518362,
         -0.03280614,  0.02737717],
        [-0.10265917,  0.02650812,  0.02271503, ..., -0.03359742,
         -0.07984945, -0.01507708],
        [ 0.02123393, -0.02468549, -0.04494374, ..., -0.10995809,
          0.0057256 ,  0.09915379],
        [ 0.01873982,  0.04382847, -0.04304255, ..., -0.07801619,
         -0.07840681, -0.00304189]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [12]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [13]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693600535392761),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.14934504032135)]

In [14]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436005115509033),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.8909372091293335)]

In [15]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [16]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['ef337d33-f0e0-4d55-85c9-45fb51bc85fb',
  '52eca65f-aa5a-492b-a796-cc707f241e31',
  '018e8cd7-4f38-48b3-98f5-8b08a63df2c9',
  '94c548ec-ff3f-428c-bea3-638a937c1f21',
  'd033685e-12e3-468a-bf1b-4d44d627b1c3'],
 'embeddings': array([[ 0.00994723,  0.06914338, -0.05147114, ..., -0.0354334 ,
          0.01284808,  0.01248293],
        [ 0.00127743,  0.03129851, -0.02375379, ..., -0.00518362,
         -0.03280614,  0.02737717],
        [-0.10265917,  0.02650812,  0.02271503, ..., -0.03359742,
         -0.07984945, -0.01507708],
        [ 0.02123393, -0.02468549, -0.04494374, ..., -0.10995809,
          0.0057256 ,  0.09915379],
        [ 0.01873982,  0.04382847, -0.04304255, ..., -0.07801619,
         -0.07840681, -0.00304189]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [17]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [18]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['ef337d33-f0e0-4d55-85c9-45fb51bc85fb',
  '52eca65f-aa5a-492b-a796-cc707f241e31',
  '018e8cd7-4f38-48b3-98f5-8b08a63df2c9',
  '94c548ec-ff3f-428c-bea3-638a937c1f21',
  'd033685e-12e3-468a-bf1b-4d44d627b1c3'],
 'embeddings': array([[ 0.00994723,  0.06914338, -0.05147114, ..., -0.0354334 ,
          0.01284808,  0.01248293],
        [ 0.00127743,  0.03129851, -0.02375379, ..., -0.00518362,
         -0.03280614,  0.02737717],
        [-0.10265917,  0.02650812,  0.02271503, ..., -0.03359742,
         -0.07984945, -0.01507708],
        [ 0.02123393, -0.02468549, -0.04494374, ..., -0.10995809,
          0.0057256 ,  0.09915379],
        [ 0.01873982,  0.04382847, -0.04304255, ..., -0.07801619,
         -0.07840681, -0.00304189]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo